In [ ]:
import os
import glob
from bs4 import BeautifulSoup
from collections import Counter
import numpy as np

CORPUS_DIR = "../corpus/arxiv"

def analyze_corpus_structure(corpus_dir):
    search_path = os.path.join(corpus_dir, "**", "article.html")
    files = glob.glob(search_path, recursive=True)
    
    print(f"Trovati {len(files)} articoli da analizzare.\n")
    
    stats = {
        "has_article_tag": 0,
        "has_ltx_page_main": 0,
        "common_classes": Counter(),
        "section_titles": Counter(),
        "structure_depth": [], # Profondità media dell'albero DOM
        "real_text_lengths": [], # Lunghezza testo puro (senza HTML)
        "bibliography_detected": 0
    }
    
    for file_path in files:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                soup = BeautifulSoup(f, 'html.parser')
                
                # 1. Verifica container principale
                if soup.find('article'):
                    stats["has_article_tag"] += 1
                
                # ar5iv usa spesso div class="ltx_page_main"
                main_div = soup.find('div', class_='ltx_page_main')
                if main_div:
                    stats["has_ltx_page_main"] += 1
                
                # Se non c'è main_div, usiamo il body per le statistiche generali
                root = main_div if main_div else soup.body
                
                if not root:
                    continue

                # 2. Analisi delle classi CSS più comuni (per capire come selezionare i paragrafi)
                # Cerchiamo tutti i tag con class
                for tag in root.find_all(True):
                    if tag.get('class'):
                        # 'class' è una lista in BS4
                        for cls in tag.get('class'):
                            stats["common_classes"][cls] += 1
                            
                # 3. Analisi delle Sezioni (Titoli)
                # In LaTeXML i titoli sono spesso in h1, h2, ecc o classi ltx_title
                headers = root.find_all(['h2', 'h3'], class_='ltx_title')
                for h in headers:
                    title_text = h.get_text().strip().lower()
                    stats["section_titles"][title_text] += 1
                    
                # 4. Rilevamento Bibliografia
                # Solitamente sezione con classe 'ltx_bibliography'
                if root.find(class_='ltx_bibliography') or root.find(attrs={"role": "doc-bibliography"}):
                    stats["bibliography_detected"] += 1

                # 5. Lunghezza effettiva del contenuto (Text-only)
                text_content = root.get_text(" ", strip=True)
                stats["real_text_lengths"].append(len(text_content))

        except Exception as e:
            print(f"Errore leggendo {file_path}: {e}")

    return stats

def print_report(stats):
    print("--- REPORT ANALISI STRUTTURALE ---")
    print(f"Articoli con tag <article>: {stats['has_article_tag']}")
    print(f"Articoli con container .ltx_page_main: {stats['has_ltx_page_main']}")
    print(f"Bibliografia rilevata (classe ltx_bibliography): {stats['bibliography_detected']}")
    
    print("\n--- Classi CSS più frequenti (Top 10) ---")
    # Queste ci dicono cosa estrarre. Es: ltx_p, ltx_section, ltx_Math
    for cls, count in stats['common_classes'].most_common(10):
        print(f"  .{cls}: {count}")

    print("\n--- Titoli di sezione più frequenti (Top 10) ---")
    # Utile per capire se "Introduction" o "Related Work" sono standard
    for title, count in stats['section_titles'].most_common(10):
        print(f"  '{title}': {count}")

    if stats["real_text_lengths"]:
        lens = stats["real_text_lengths"]
        print("\n--- Statistiche Lunghezza Testo Puro (Caratteri) ---")
        print(f"  Min: {np.min(lens)}")
        print(f"  Max: {np.max(lens)}")
        print(f"  Media: {np.mean(lens):.0f}")
        print(f"  Mediana: {np.median(lens):.0f}")

if __name__ == "__main__":
    stats = analyze_corpus_structure(CORPUS_DIR)
    print_report(stats)

Trovati 536 articoli da analizzare.

--- REPORT ANALISI STRUTTURALE ---
Articoli con tag <article>: 536
Articoli con container .ltx_page_main: 536
Bibliografia rilevata (classe ltx_bibliography): 531

--- Classi CSS più frequenti (Top 10) ---
  .ltx_text: 409311
  .ltx_font_typewriter: 230728
  .ltx_td: 172837
  .ltx_align_center: 101484
  .ltx_lst_space: 96389
  .ltx_lst_identifier: 85126
  .ltx_p: 64575
  .ltx_bibblock: 64368
  .ltx_ref: 63473
  .ltx_Math: 58751

--- Titoli di sezione più frequenti (Top 10) ---
  'references': 534
  '1 introduction': 381
  '2 related work': 164
  '6 conclusion': 97
  '4 experiments': 93
  '1. introduction': 85
  'limitations': 80
  '5 conclusion': 63
  '7 conclusion': 62
  'acknowledgments': 59

--- Statistiche Lunghezza Testo Puro (Caratteri) ---
  Min: 11360
  Max: 287670
  Media: 61922
  Mediana: 53976


In [ ]:
# scarica senza secondo filtro e controlla quanti pagine non hanno i temrini cercati
# controlla i tag figure che classi hanno implementando una logica più stretta per tables.
# poi vai su pubmed

In [1]:
import os, json
datapath = "../../corpus/pubmed/"
for d in os.listdir(datapath):
    metadata = os.path.join(datapath, d, "metadata.json")
    with open(metadata, "r") as f:
        md = json.load(f)
    if md["summary"]== None:
        print(d)
'''
da mettere nel report:
conoscenza della logica estrazione abstract debole, basata su section class="abstract". 
per testare la solidità di questo campo e analizzare quali alltri tag classi con attributi contenessero l'abstract stampa il nome e analizza manualmete.
dall'analisi manuale è emerso che x documenti non hanno <section class="abstract"...> o ce l'hanno ma vuoto. 
A seconda conferma quando si va sul sito di pubmed nel campo abstract compare No abstract available (https://pubmed.ncbi.nlm.nih.gov/40882668/)
conseguenza la logica <section class="abstract"> è forte perlomeno per questo campione di documenti pubmed
'''

10225660
10719848
10928813
11409641
11628486
12396810
12561744
1976674
2376922
3114335
3796673
4981751
6165739
6406069
6589848
6796246
6836928
6860002
7173517
7495170
7658457
7672678
7705438
7720815
7768980
8146094
9013624
9127917
9396106
9623052
9909694


'\nda mettere nel report:\nconoscenza della logica estrazione abstract debole, basata su section class="abstract". \nper testare la solidità di questo campo e analizzare quali alltri tag classi con attributi contenessero l\'abstract stampa il nome e analizza manualmete.\ndall\'analisi manuale è emerso che x documenti non hanno <section class="abstract"...> o ce l\'hanno ma vuoto. \nA seconda conferma quando si va sul sito di pubmed nel campo abstract compare No abstract available (https://pubmed.ncbi.nlm.nih.gov/40882668/)\nconseguenza la logica <section class="abstract"> è forte perlomeno per questo campione di documenti pubmed\n'